In [0]:
dbutils.widgets.text("BASE_URL", "https://api.spotify.com/v1")
dbutils.widgets.text("TOKEN_URL", "https://accounts.spotify.com/api/token")
dbutils.widgets.text("AUTH_URL", "https://accounts.spotify.com/authorize")
dbutils.widgets.text("PlaylistId", "")

In [0]:
client_secret = dbutils.secrets.get("spotify-secrets", "client_secret")
client_id = dbutils.secrets.get("spotify-secrets", "client_id")
access_token = dbutils.secrets.get("spotify-secrets", "access_token")
refresh_token = dbutils.secrets.get("spotify-secrets", "refresh_token")

redirect_uri = '"http://localhost:8888/callback"'


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret("spotify-secrets", "client_secret", string_value="f1bfab69717f489dbc183bff7f142fe0")
w.secrets.put_secret("spotify-secrets", "client_id", string_value="b6076a3c3b224b25b4a9beb61b92d596")
w.secrets.put_secret("spotify-secrets", "access_token", string_value="BQD29SpmvyPNHiNlkZUFnfK8-4-doGbF7nTXStaWQFp9js_apo1Hfr9SEmwU-PeNuCGi_jZHUytGtct5cG1sWBxSGDnZGKU6AefoLyvqB7V1q6QK3DFldQgQ1E_coCs2-ucwE0P4rqt2NEm8TjOsUftxDBRMh7W5LPO69DB5sgFBR-X47xkMgBJdN2eTdhkrXBDiask2_yGj4ucD8lj72kdwaL2SUxhBMPVJ-Mcu")
w.secrets.put_secret("spotify-secrets", "refresh_token", string_value="AQAvkO13XCi26w_YtFyderu71Fkw9TN8i39StEc72QpgITkhsLZHrVVIE2dJfWOxmiYroA4o8KrYPR25pCtCH09VzsRUVP1RSXYakt9F1zBzPMLJ_vtpnkWOHpnVC8UJVQE")

In [0]:
# from pyspark.sql import SparkSession
# import pyspark.sql.functions as F
# import requests
# import json
# import base64

# class SpotifyAPI:
#   def __init__(self):
#     self.client_id = client_id
#     self.client_secret = client_secret
#     self.base_url = dbutils.widgets.get("BASE_URL")
#     self.token_url = dbutils.widgets.get("TOKEN_URL")
#     self.auth_url = dbutils.widgets.get("AUTH_URL")
  
#   def get_access_token(self):
#       # try:

#         auth_response_code = requests.get(
#           self.auth_url, {
#             'client_id': self.client_id,
#             'response_type': 'code',
#             'redirect_uri': '"http://localhost:8888/callback"'
#           }
#         )
        
#         print(auth_response_code)

#         data = {
#           'grant_type': 'authorization_code',
#           'code': str(auth_response_code),
#           'redirect_uri': '"http://localhost:8888/callback"',

#         }
#         auth_str = f"{self.client_id}:{self.client_secret}"
#         auth_header = base64.b64encode(auth_str.encode()).decode()

#         headers = {
#           'authorization': f"Basic {auth_header}",
#           'content-type': 'application/x-www-form-urlencoded'
#         }

#         response = requests.post(
#           self.token_url,
#           data = data,
#           headers = headers
#         )

#         print(response.json())


#       # except:
#       #   print('something went wrong')

# if __name__ == "__main__":
#   api = SpotifyAPI()
#   api.get_access_token()
  
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import json
import requests
import base64
from urllib.parse import urlencode
import webbrowser

class SpotifyAPI:
    def __init__(self):
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url =  dbutils.widgets.get("BASE_URL")
        self.auth_url =  dbutils.widgets.get("AUTH_URL")
        self.token_url =  dbutils.widgets.get("TOKEN_URL")
        self.redirect_uri = "http://127.0.0.1:8888/callback"

    # only need to run once, run get_valid_access_token afterwards
    def authorize_once(self):
        params = {
            "client_id": self.client_id,
            "response_type": "code",
            "redirect_uri": self.redirect_uri,
            "scope": "playlist-read-private"
        }

        auth_request_url = f"{self.auth_url}?{urlencode(params)}"

        print("\nOpen this URL in your browser:\n")
        print(auth_request_url)

        auth_code = input("\nPaste the code from the callback URL here:\n")

        auth_str = f"{self.client_id}:{self.client_secret}"
        auth_header = base64.b64encode(auth_str.encode()).decode()

        headers = {
            "Authorization": f"Basic {auth_header}",
            "Content-Type": "application/x-www-form-urlencoded"
        }

        data = {
            "grant_type": "authorization_code",
            "code": auth_code,
            "redirect_uri": self.redirect_uri
        }

        response = requests.post(
            self.token_url,
            headers=headers,
            data=data
        )

        response_data = response.json()

        access_token = response_data['access_token']
        refresh_token = response_data['refresh_token']

        dbutils.secrets.put_secret("spotify-secrets", "access_token", string_value=access_token)
        dbutils.secrets.put_secret("spotify-secrets", "refresh_token", string_value=refresh_token)

    def refresh_access_token:
      refresh_token = dbutils.secrets.get("spotify-secrets", "refresh_token")
      headers = { "Content-Type": "application/x-www-form-urlencoded" }
      data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token
      }

      response = requests.post(
        self.token_url,
        headers = headers,
        data = data
      )

    def get_valid_access_token:


if __name__ == "__main__":
    api = SpotifyAPI()
    api.get_access_token()